# Run script as a command job

The goal of this notebook is to run the training script in `src/` as a command job using the python SDK.

In [1]:
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient, command, Input

In [2]:
try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")
except Exception as ex:
    credential = InteractiveBrowserCredential()
    



In [11]:
ml_client = MLClient.from_config(credential=credential, path="../config.json")

Found the config file in: ../config.json


To specify a job, add the run information as parameters to the command function.
The same parameters will later be used to create a `job.yaml`.
Of note: a prebuilt environment is used here to visualize functionality.
In a production environment, a custom docker image should be created from the requirements and pushed to ACR.

In [15]:
job = command(
    code="../src",
    command="python train_model.py --data ${{inputs.training_data}}",
    inputs={
        "training_data": Input(type="uri_file", path="../data/Student_Performance.csv")
    },
    compute="cluster-mlops",
    environment="AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
    display_name="student_performance-command-job",
    experiment_name="student_performance"
)

job_ref = ml_client.create_or_update(job)
print(f"job is running at: {job_ref.studio_url}")

Uploading src (0.0 MBs): 100%|██████████| 1965/1965 [00:00<00:00, 69484.86it/s]




job is running at: https://ml.azure.com/runs/olden_ship_q73nkkylf6?wsid=/subscriptions/34503611-a3d3-454f-806e-1abc48aeba67/resourcegroups/rg-mlops-0a0ad1f0eeee45bf86/workspaces/mlw-mlops-0a0ad1f0eeee45bf86&tid=f5b0ba10-7374-4ab8-ae85-408826951cc0
